---
## Cell 0 — Поиск гиперпараметров (11 экспериментов)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../src").resolve()))

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import time

from src.transforms import get_transforms
from src.data import prepare_loaders
from src.models.resnet import create_resnet18
from src.training import train_one_epoch, validate, step_scheduler, save_history
from src.utils import set_seed

# ==========================================
# 🔧 КОНФИГУРАЦИЯ
# ==========================================
DATA_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")
SAVE_DIR = Path(r"./night_results")
SAVE_DIR.mkdir(exist_ok=True)

MODALITIES = ["ДС", "УФ"]
EPOCHS = 10
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPERIMENTS = [
    {"id": "0_Reference",    "lr": 1e-4, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",  "aug": "std"},
    {"id": "1_LR_Low",       "lr": 5e-5, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",  "aug": "std"},
    {"id": "2_LR_High",      "lr": 5e-4, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",  "aug": "std"},
    {"id": "3_WD_Low",       "lr": 1e-4, "wd": 1e-5, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",  "aug": "std"},
    {"id": "4_WD_High",      "lr": 1e-4, "wd": 1e-3, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",  "aug": "std"},
    {"id": "5_Drop_Low",     "lr": 1e-4, "wd": 1e-4, "dropout": 0.3, "opt": "adamw", "sched": "plateau", "resize": "pad",  "aug": "std"},
    {"id": "6_Drop_High",    "lr": 1e-4, "wd": 1e-4, "dropout": 0.7, "opt": "adamw", "sched": "plateau", "resize": "pad",  "aug": "std"},
    {"id": "7_Opt_SGD",      "lr": 1e-4, "wd": 1e-4, "dropout": 0.5, "opt": "sgd",   "sched": "plateau", "resize": "pad",  "aug": "std"},
    {"id": "8_Sched_Cosine", "lr": 1e-4, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "cosine",  "resize": "pad",  "aug": "std"},
    {"id": "9_Aug_Heavy",    "lr": 1e-4, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "pad",  "aug": "heavy"},
    {"id": "10_Resize_Crop", "lr": 1e-4, "wd": 1e-4, "dropout": 0.5, "opt": "adamw", "sched": "plateau", "resize": "crop", "aug": "std"},
]


# ==========================================
# 🧪 ЗАПУСК ЭКСПЕРИМЕНТА
# ==========================================
def run_experiment(modality, cfg):
    set_seed(42)
    print(f"\n🚀 {cfg['id']} | {modality} | LR:{cfg['lr']:.0e} WD:{cfg['wd']:.0e} Drop:{cfg['dropout']} Opt:{cfg['opt']} Sched:{cfg['sched']}")
    print("-" * 80)

    train_loader, val_loader, classes = prepare_loaders(
        DATA_ROOT / modality,
        train_transform=get_transforms(cfg["resize"], cfg["aug"], is_train=True),
        val_transform=get_transforms(cfg["resize"], cfg["aug"], is_train=False),
        batch_size=BATCH_SIZE,
    )
    model = create_resnet18(len(classes), dropout_p=cfg["dropout"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    if cfg["opt"] == "adamw":
        optimizer = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    else:
        optimizer = optim.SGD(model.parameters(), lr=cfg["lr"], momentum=0.9, weight_decay=cfg["wd"])

    if cfg["sched"] == "plateau":
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    else:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_metrics = {"f1": 0.0, "epoch": 0}
    history = []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, epoch, EPOCHS)
        val_mets = validate(model, val_loader, criterion, DEVICE, epoch, EPOCHS)
        step_scheduler(scheduler, cfg["sched"], metric=val_mets["f1"])

        if val_mets["f1"] > best_metrics["f1"]:
            best_metrics = {**val_mets, "epoch": epoch}
            torch.save(model.state_dict(), SAVE_DIR / f"{cfg['id']}_{modality}_best.pth")

        print(f"✅ Ep {epoch:2d} | TrL:{train_loss:.4f} | VL:{val_mets['loss']:.4f} F1:{val_mets['f1']:.4f} Acc:{val_mets['acc']:.4f} | {time.time()-t0:.1f}s")
        history.append({"epoch": epoch, **val_mets})

    save_history(history, SAVE_DIR / f"{cfg['id']}_{modality}_history.json")
    torch.cuda.empty_cache()
    return {"config": cfg, "modality": modality, "best": best_metrics}


# ==========================================
# 📊 ЗАПУСК + СВОДНАЯ ТАБЛИЦА
# ==========================================
print(f"🖥️ Device: {DEVICE} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}")

all_results = []
for mod in MODALITIES:
    for cfg in EXPERIMENTS:
        all_results.append(run_experiment(mod, cfg))

rows = []
for r in all_results:
    c, m, b = r["config"], r["modality"], r["best"]
    rows.append({
        "Model": c["id"], "Modality": m,
        "LR": f"{c['lr']:.0e}", "WD": f"{c['wd']:.0e}", "Dropout": c["dropout"],
        "Optimizer": c["opt"], "Scheduler": c["sched"], "Resize": c["resize"], "Aug": c["aug"],
        "Best_Epoch": b["epoch"],
        "Val_F1": b["f1"], "Val_Acc": b["acc"], "Val_Prec": b["prec"], "Val_Rec": b["rec"], "Val_Loss": b["loss"],
    })
df = pd.DataFrame(rows)

for mod in MODALITIES:
    ref = df[(df["Model"] == "0_Reference") & (df["Modality"] == mod)].iloc[0]
    mask = df["Modality"] == mod
    for col, ref_col in [("ΔF1","Val_F1"), ("ΔAcc","Val_Acc"), ("ΔPrec","Val_Prec"), ("ΔRec","Val_Rec"), ("ΔLoss","Val_Loss")]:
        df.loc[mask, col] = (df.loc[mask, ref_col] - ref[ref_col]).round(4)

df = df.sort_values(["Modality", "Model"])
display_cols = ["Model", "Modality", "LR", "WD", "Dropout", "Optimizer", "Scheduler", "Resize", "Aug",
                "Val_F1", "ΔF1", "Val_Acc", "ΔAcc", "Val_Prec", "ΔPrec", "Val_Rec", "ΔRec", "Val_Loss", "ΔLoss"]

pd.options.display.max_columns = None
pd.options.display.width = 200
print("\n" + "="*120)
print("📊 СРАВНИТЕЛЬНАЯ ТАБЛИЦА (Δ = отклонение от 0_Reference)")
print("="*120)
print(df[display_cols].to_string(index=False))
df.to_csv(SAVE_DIR / "full_comparison.csv", index=False)
print(f"\n💾 {SAVE_DIR / 'full_comparison.csv'}")


---
## Cell 1 — Эксперименты с заморозкой слоёв

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../src").resolve()))

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

from src.transforms import get_transforms
from src.data import prepare_loaders
from src.models.resnet import create_resnet18
from src.training import train_one_epoch, validate, EarlyStopping

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")
SAVE_DIR = Path(r"./freeze_test_best")
SAVE_DIR.mkdir(exist_ok=True)

PATIENCE = 5
MIN_DELTA = 0.005
BATCH_SIZE = 64

EXPERIMENTS = [
    {"id": "A_DS_None",    "mod": "ДС", "freeze": "none",    "lr": 5e-5, "wd": 1e-3},
    {"id": "B_DS_Partial", "mod": "ДС", "freeze": "partial", "lr": 5e-5, "wd": 1e-3},
    {"id": "C_DS_Full",    "mod": "ДС", "freeze": "full",    "lr": 1e-3, "wd": 1e-3},
    {"id": "D_UF_None",    "mod": "УФ", "freeze": "none",    "lr": 1e-4, "wd": 1e-5},
    {"id": "E_UF_Partial", "mod": "УФ", "freeze": "partial", "lr": 1e-4, "wd": 1e-5},
    {"id": "F_UF_Full",    "mod": "УФ", "freeze": "full",    "lr": 1e-3, "wd": 1e-5},
]


def run_experiment(cfg):
    print(f"\n🚀 {cfg['id']} | Freeze: {cfg['freeze']} | {cfg['mod']}")

    train_loader, val_loader, classes = prepare_loaders(
        DATA_ROOT / cfg["mod"],
        train_transform=get_transforms("pad", "std", is_train=True),
        val_transform=get_transforms("pad", "std", is_train=False),
        batch_size=BATCH_SIZE,
    )
    model = create_resnet18(len(classes), freeze_mode=cfg["freeze"], dropout_p=0.5).to(DEVICE)
    optimizer = optim.SGD(model.parameters(), lr=cfg["lr"], momentum=0.9, weight_decay=cfg["wd"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    criterion = nn.CrossEntropyLoss()
    es = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

    for epoch in range(1, 51):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, epoch, 50)
        val_mets = validate(model, val_loader, criterion, DEVICE, epoch, 50)
        scheduler.step()

        improved = es.step(val_mets["f1"], epoch)
        if improved:
            torch.save(model.state_dict(), SAVE_DIR / f"{cfg['id']}_best.pth")

        print(f"✅ Ep {epoch:2d} | TrL: {train_loss:.4f} | VL: {val_mets['loss']:.4f} | F1: {val_mets['f1']:.4f} {es.status}")
        if es.should_stop:
            print(f"⏹️ Early Stop at epoch {epoch}. Best F1: {es.best:.4f}")
            break

    torch.cuda.empty_cache()
    return {"id": cfg["id"], "modality": cfg["mod"], "freeze": cfg["freeze"],
            "best_f1": round(es.best, 4), "best_epoch": es.best_epoch}


print(f"🖥️ Device: {DEVICE}")
results = [run_experiment(cfg) for cfg in EXPERIMENTS]

df = pd.DataFrame(results)
print("\n" + "="*80 + "\n📊 ИТОГИ\n" + "="*80)
for mod in ["ДС", "УФ"]:
    print(f"\n🔹 Модальность: {mod}")
    print(df[df["modality"] == mod].sort_values("best_f1", ascending=False)[["id","freeze","best_f1","best_epoch"]].to_string(index=False))

df.to_csv(SAVE_DIR / "freeze_comparison.csv", index=False)
print(f"\n💾 {SAVE_DIR / 'freeze_comparison.csv'}")


---
## Cell 2 — Аблационное исследование (FocalLoss / LabelSmooth / CutMix)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../src").resolve()))

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

from src.utils import set_seed
from src.transforms import get_transforms
from src.data import prepare_loaders
from src.models.resnet import create_resnet18
from src.losses import get_criterion
from src.augmentation import cutmix_data
from src.training import train_one_epoch, validate, EarlyStopping

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")
SAVE_DIR = Path(r"./ablation_strict")
SAVE_DIR.mkdir(exist_ok=True)

PATIENCE = 5
MIN_DELTA = 0.005
BATCH_SIZE = 64
MAX_EPOCHS = 50

EXPERIMENTS = [
    {"id": "DS_Base",        "mod": "ДС", "freeze": "none",    "lr": 5e-5, "wd": 1e-3, "drop": 0.3, "loss": "ce",           "mix": "none",    "aug": "heavy", "resize": "pad"},
    {"id": "DS_FocalLoss",   "mod": "ДС", "freeze": "none",    "lr": 5e-5, "wd": 1e-3, "drop": 0.3, "loss": "focal",        "mix": "none",    "aug": "heavy", "resize": "pad"},
    {"id": "DS_LabelSmooth", "mod": "ДС", "freeze": "none",    "lr": 5e-5, "wd": 1e-3, "drop": 0.3, "loss": "label_smooth", "mix": "none",    "aug": "heavy", "resize": "pad"},
    {"id": "DS_CutMix",      "mod": "ДС", "freeze": "none",    "lr": 5e-5, "wd": 1e-3, "drop": 0.3, "loss": "ce",           "mix": "cutmix",  "aug": "heavy", "resize": "pad"},
    {"id": "UF_Base",        "mod": "УФ", "freeze": "partial", "lr": 5e-4, "wd": 1e-5, "drop": 0.3, "loss": "ce",           "mix": "none",    "aug": "std",   "resize": "crop"},
    {"id": "UF_FocalLoss",   "mod": "УФ", "freeze": "partial", "lr": 5e-4, "wd": 1e-5, "drop": 0.3, "loss": "focal",        "mix": "none",    "aug": "std",   "resize": "crop"},
    {"id": "UF_LabelSmooth", "mod": "УФ", "freeze": "partial", "lr": 5e-4, "wd": 1e-5, "drop": 0.3, "loss": "label_smooth", "mix": "none",    "aug": "std",   "resize": "crop"},
    {"id": "UF_CutMix",      "mod": "УФ", "freeze": "partial", "lr": 5e-4, "wd": 1e-5, "drop": 0.3, "loss": "ce",           "mix": "cutmix",  "aug": "std",   "resize": "crop"},
]


def run_experiment(cfg):
    print(f"\n🚀 {cfg['id']} | {cfg['mod']} | Loss: {cfg['loss']} | Mix: {cfg['mix']}")
    gen = set_seed(42)

    mix_fn = cutmix_data if cfg["mix"] == "cutmix" else None

    train_loader, val_loader, classes = prepare_loaders(
        DATA_ROOT / cfg["mod"],
        train_transform=get_transforms(cfg["resize"], cfg["aug"], is_train=True),
        val_transform=get_transforms(cfg["resize"], cfg["aug"], is_train=False),
        batch_size=BATCH_SIZE,
        generator=gen,
    )
    model = create_resnet18(len(classes), freeze_mode=cfg["freeze"], dropout_p=cfg["drop"]).to(DEVICE)
    optimizer = optim.SGD(model.parameters(), lr=cfg["lr"], momentum=0.9, weight_decay=cfg["wd"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    criterion = get_criterion(cfg["loss"])
    es = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, epoch, MAX_EPOCHS, mix_fn=mix_fn)
        val_mets = validate(model, val_loader, criterion, DEVICE, epoch, MAX_EPOCHS)
        scheduler.step()

        improved = es.step(val_mets["f1"], epoch)
        if improved:
            torch.save(model.state_dict(), SAVE_DIR / f"{cfg['id']}_best.pth")

        print(f"✅ Ep {epoch:2d} | TrL: {train_loss:.4f} | VL: {val_mets['loss']:.4f} | F1: {val_mets['f1']:.4f} {es.status}")
        if es.should_stop:
            print(f"⏹️ Early Stop at {epoch}. Best F1: {es.best:.4f}")
            break

    torch.cuda.empty_cache()
    return {"id": cfg["id"], "modality": cfg["mod"], "loss": cfg["loss"], "mix": cfg["mix"],
            "best_f1": round(es.best, 4), "best_epoch": es.best_epoch}


print(f"🖥️ Device: {DEVICE}")
results = [run_experiment(cfg) for cfg in EXPERIMENTS]

df = pd.DataFrame(results)
for mod in ["ДС", "УФ"]:
    base_f1 = df[(df["id"].str.contains("Base")) & (df["modality"] == mod)]["best_f1"].values[0]
    mask = df["modality"] == mod
    df.loc[mask, "ΔF1"] = (df.loc[mask, "best_f1"] - base_f1).round(4)

df = df.sort_values(["modality", "id"])
print("\n" + "="*90 + "\n📊 ИТОГИ A/B ТЕСТИРОВАНИЯ\n" + "="*90)
for mod in ["ДС", "УФ"]:
    print(f"\n🔹 {mod} (База ΔF1 = 0.0000)")
    print(df[df["modality"] == mod][["id","loss","mix","best_f1","ΔF1","best_epoch"]].to_string(index=False))

df.to_csv(SAVE_DIR / "ablation_strict.csv", index=False)
print(f"\n💾 {SAVE_DIR / 'ablation_strict.csv'}")


---
## Cell 3 — Class-Aware Augmentation

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../src").resolve()))

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

from src.utils import set_seed
from src.transforms import get_transforms
from src.data import ClassAwareImageFolder, make_weighted_sampler, prepare_loaders
from src.models.resnet import create_resnet18
from src.training import train_one_epoch, validate, EarlyStopping
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")
SAVE_DIR = Path(r"./class_aug_test_light_vs_heavy")
SAVE_DIR.mkdir(exist_ok=True)

PATIENCE = 5
MIN_DELTA = 0.005
BATCH_SIZE = 64
MAX_EPOCHS = 50

SMALL_CLASSES = [
    "Уголь,_уголь_с_прослоями_аргиллита",
    "Песчаник_с_прослоями_аргиллита",
    "Глинисто-карбонатная_порода",
    "Алевролит",
]

EXPERIMENTS = [
    {"id": "DS_StdAll",   "mod": "ДС", "class_aware": False},
    {"id": "DS_ClassAug", "mod": "ДС", "class_aware": True},
    {"id": "UF_StdAll",   "mod": "УФ", "class_aware": False},
    {"id": "UF_ClassAug", "mod": "УФ", "class_aware": True},
]


def class_aware_transform_fn(class_name: str, class_aware: bool):
    """Возвращает Compose-пайплайн с учётом принадлежности к малому классу."""
    aug = "heavy" if (class_aware and class_name in SMALL_CLASSES) else "light"
    return get_transforms("pad", aug, is_train=True)


def run_experiment(cfg):
    print(f"\n🚀 {cfg['id']} | {cfg['mod']} | ClassAug: {cfg['class_aware']}")
    gen = set_seed(42)

    mod_path = DATA_ROOT / cfg["mod"]
    lr = 5e-5 if cfg["mod"] == "ДС" else 5e-4
    wd = 1e-3  if cfg["mod"] == "ДС" else 1e-5
    freeze = "none" if cfg["mod"] == "ДС" else "partial"

    # Кастомный датасет с per-class аугментацией
    train_ds = ClassAwareImageFolder(
        mod_path / "train",
        transform_fn=lambda cls, aware: class_aware_transform_fn(cls, aware),
        class_aware=cfg["class_aware"],
        is_train=True,
    )
    val_transform = get_transforms("pad", "none", is_train=False)
    val_ds_raw = __import__("torchvision").datasets.ImageFolder(mod_path / "val", transform=val_transform)

    sampler = make_weighted_sampler(train_ds.targets, generator=gen)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=0, pin_memory=False, generator=gen)
    val_loader = DataLoader(val_ds_raw, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, pin_memory=False)

    model = create_resnet18(len(train_ds.classes), freeze_mode=freeze, dropout_p=0.3).to(DEVICE)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    criterion = nn.CrossEntropyLoss()
    es = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, epoch, MAX_EPOCHS)
        val_mets = validate(model, val_loader, criterion, DEVICE, epoch, MAX_EPOCHS)
        scheduler.step()

        improved = es.step(val_mets["f1"], epoch)
        if improved:
            torch.save(model.state_dict(), SAVE_DIR / f"{cfg['id']}_best.pth")

        print(f"✅ Ep {epoch:2d} | TrL: {train_loss:.4f} | VL: {val_mets['loss']:.4f} | F1: {val_mets['f1']:.4f} {es.status}")
        if es.should_stop:
            print(f"⏹️ Early Stop at {epoch}. Best F1: {es.best:.4f}")
            break

    torch.cuda.empty_cache()
    return {"id": cfg["id"], "modality": cfg["mod"], "class_aware": cfg["class_aware"],
            "best_f1": round(es.best, 4), "best_epoch": es.best_epoch}


print(f"🖥️ Device: {DEVICE}")
results = [run_experiment(cfg) for cfg in EXPERIMENTS]

df = pd.DataFrame(results)
for mod in ["ДС", "УФ"]:
    base_f1 = df[(df["id"].str.contains("StdAll")) & (df["modality"] == mod)]["best_f1"].values[0]
    mask = df["modality"] == mod
    df.loc[mask, "ΔF1"] = (df.loc[mask, "best_f1"] - base_f1).round(4)

df = df.sort_values(["modality", "id"])
print("\n" + "="*90 + "\n📊 ИТОГИ: Light vs Heavy Augmentation\n" + "="*90)
for mod in ["ДС", "УФ"]:
    print(f"\n🔹 {mod}")
    print(df[df["modality"] == mod][["id","class_aware","best_f1","ΔF1"]].to_string(index=False))

df.to_csv(SAVE_DIR / "class_aug_light_heavy.csv", index=False)
print(f"\n💾 {SAVE_DIR / 'class_aug_light_heavy.csv'}")
